In [1]:
import os
os.chdir('application')

In [2]:

# from typing import List
# from pydantic import BaseModel
import redis

from modules.redis import RedisConversationMemory
# from modules.retriever_wrappper import SanitizingRetriever
from modules.history_utils import build_chat_history_messages

from config import (REDIS_URL, MAX_HISTORY_MESSAGES, 
                    gemini_chat_model, google_api_key_name, qa_system_prompt
)
from vectorstore.milvus_vectorstore import get_vectorstore

# import logging

# LangChain imports (use provider wrapper)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.output_parsers import PydanticOutputParser

C:\Users\PAVILION\miniconda3\envs\rag-chatbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from schemas.rag_outputs import AnswerWithSources

In [4]:
# from langchain_core.runnables import RunnablePassthrough
# from langchain_core.output_parsers import StrOutputParser
# from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

In [5]:
from schemas.rag_outputs import AnswerWithSources, format_docs_with_metadata

In [6]:
# ----- CONFIG -----
r = redis.Redis.from_url(REDIS_URL, decode_responses=True)
# app = FastAPI()

In [7]:
# Your already-created vectorstore (LangChain Milvus wrapper)
vectorstore = get_vectorstore()

2025-10-14 10:50:00,827 - vectorstore.milvus_vectorstore - INFO - Connecting to Milvus
2025-10-14 10:50:01,057 - vectorstore.milvus_vectorstore - INFO - Initialised Milvus vector store


In [8]:
mem = RedisConversationMemory(session_id='002', max_messages=MAX_HISTORY_MESSAGES)


In [9]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

In [10]:
llm = ChatGoogleGenerativeAI(model=gemini_chat_model, google_api_key=os.environ.get(google_api_key_name))

In [11]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt+'ini adalah sesi debugging. Tambahkan retrieval text pada field "answer"'),
    MessagesPlaceholder(variable_name="chat_history"),            # inject conversation history here
#     ("system", "Gunakan dokumen yang diekstrak berikut untuk menjawab pertanyaan user:\n\n{context}"),
    ("human", "{question}")
])

In [12]:
pydantic_parser = PydanticOutputParser(pydantic_object=AnswerWithSources)

In [13]:
structured_llm = llm.with_structured_output(AnswerWithSources)

In [14]:
rag_chain = (
    {
        # The retriever runs first, using the original question.
        # Its output is passed to format_docs and becomes the 'context'.
        "context": (lambda x: x["question"]) | retriever | format_docs_with_metadata,
         
        # The original 'question' is passed through unchanged.
        "question": (lambda x: x["question"]),
        
        # The 'chat_history' is passed through unchanged.
        "chat_history": (lambda x: x["chat_history"])
    }
    | qa_prompt
    | llm
    | pydantic_parser
)

In [15]:
raw_history = mem.get_chat_history_pairs() 
raw_history

[('Halo, nama saya Henry golongan 6. Salam kenal. Apa yang bisa anda bantu?',
  'Halo Henry, salam kenal juga. Saya Nastari, asisten AI HR di KPN-Corp yang berlokasi di Kuningan, Jakarta. Saya di sini untuk membantu Anda dengan pertanyaan terkait *kebijakan perusahaan* dan *masalah karyawan*. Jangan ragu untuk bertanya jika ada yang ingin Anda ketahui.'),
 ('Halo, nama saya Henry golongan 6. Salam kenal. Apa yang bisa anda bantu?',
  ''),
 ('Halo, nama saya Henry golongan 6. Salam kenal. Apa yang bisa anda bantu?',
  '')]

In [16]:
question = 'Halo, nama saya Henry golongan 6. Salam kenal. Apa yang bisa anda bantu?'

In [17]:
question1 = "Boleh saya bertanya, Nastari? Apakah saya bisa cuti apabila istri saya melahirkan?"

In [18]:
question2 = "Jelaskan lebih detail"

In [19]:
chat_history_messages = build_chat_history_messages(raw_history)
chat_history_messages

[HumanMessage(content='Halo, nama saya Henry golongan 6. Salam kenal. Apa yang bisa anda bantu?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Halo Henry, salam kenal juga. Saya Nastari, asisten AI HR di KPN-Corp yang berlokasi di Kuningan, Jakarta. Saya di sini untuk membantu Anda dengan pertanyaan terkait *kebijakan perusahaan* dan *masalah karyawan*. Jangan ragu untuk bertanya jika ada yang ingin Anda ketahui.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Halo, nama saya Henry golongan 6. Salam kenal. Apa yang bisa anda bantu?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Halo, nama saya Henry golongan 6. Salam kenal. Apa yang bisa anda bantu?', additional_kwargs={}, response_metadata={})]

In [20]:
# The invoke call now passes a single dictionary with all the required keys
result = rag_chain.invoke({
    "question": question1, 
    "chat_history": chat_history_messages
})
result

AnswerWithSources(answer='Halo Henry, tentu saja Anda boleh bertanya. Terkait pertanyaan Anda, *ya*, Anda bisa mengajukan cuti apabila istri Anda melahirkan.\n\nKebijakan perusahaan kami mencakup:\n*   **Istri Karyawan Melahirkan/Keguguran** termasuk dalam **Jenis-Jenis Izin Khusus** yang disediakan oleh perusahaan.\n\n**Catatan:** Untuk detail lebih lanjut mengenai prosedur pengajuan, durasi cuti, atau syarat lainnya, silakan hubungi tim HRD asli atau merujuk pada dokumen kebijakan cuti yang relevan.', sources=[Source(filename='N/A', page=None), Source(filename='N/A', page=None)])

In [21]:
result.answer

'Halo Henry, tentu saja Anda boleh bertanya. Terkait pertanyaan Anda, *ya*, Anda bisa mengajukan cuti apabila istri Anda melahirkan.\n\nKebijakan perusahaan kami mencakup:\n*   **Istri Karyawan Melahirkan/Keguguran** termasuk dalam **Jenis-Jenis Izin Khusus** yang disediakan oleh perusahaan.\n\n**Catatan:** Untuk detail lebih lanjut mengenai prosedur pengajuan, durasi cuti, atau syarat lainnya, silakan hubungi tim HRD asli atau merujuk pada dokumen kebijakan cuti yang relevan.'

In [22]:
assistant_text = result.answer or result.result or ""
sources_meta = result.sources or []
# sources_meta = [d for d in source_docs]

assistant_text, sources_meta

('Halo Henry, tentu saja Anda boleh bertanya. Terkait pertanyaan Anda, *ya*, Anda bisa mengajukan cuti apabila istri Anda melahirkan.\n\nKebijakan perusahaan kami mencakup:\n*   **Istri Karyawan Melahirkan/Keguguran** termasuk dalam **Jenis-Jenis Izin Khusus** yang disediakan oleh perusahaan.\n\n**Catatan:** Untuk detail lebih lanjut mengenai prosedur pengajuan, durasi cuti, atau syarat lainnya, silakan hubungi tim HRD asli atau merujuk pada dokumen kebijakan cuti yang relevan.',
 [Source(filename='N/A', page=None), Source(filename='N/A', page=None)])

In [26]:
sources_meta[0].filename

'N/A'

In [23]:
assistant_text

'Halo Henry, tentu saja Anda boleh bertanya. Terkait pertanyaan Anda, *ya*, Anda bisa mengajukan cuti apabila istri Anda melahirkan.\n\nKebijakan perusahaan kami mencakup:\n*   **Istri Karyawan Melahirkan/Keguguran** termasuk dalam **Jenis-Jenis Izin Khusus** yang disediakan oleh perusahaan.\n\n**Catatan:** Untuk detail lebih lanjut mengenai prosedur pengajuan, durasi cuti, atau syarat lainnya, silakan hubungi tim HRD asli atau merujuk pada dokumen kebijakan cuti yang relevan.'

In [24]:
# 10) Save the turn into Redis after successful call: user then assistant
mem.add_user_message(question)
mem.add_assistant_message(assistant_text, sources=sources_meta)

2025-10-14 10:50:11,203 - modules.redis - INFO - Stored user message


TypeError: Object of type Source is not JSON serializable

In [ ]:
raw_history = mem.get_chat_history_pairs() 
raw_history

In [ ]:
stop

In [ ]:
chat_history_messages = build_chat_history_messages(raw_history)
chat_history_messages

In [ ]:
question2 = "Siapakah nama saya tadi?"

In [ ]:
# The invoke call now passes a single dictionary with all the required keys
result = rag_chain.invoke({
    "question": question2, 
    "chat_history": chat_history_messages
})
result

In [ ]:
stop

In [ ]:
conv_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=None,  # ok if you pass chat_history manually
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": qa_prompt},
)

In [ ]:
result = conv_chain.invoke({"question": question, "chat_history": chat_history_messages})
result

In [ ]:
from langchain.chains import (
    create_history_aware_retriever,
)
from langchain.chains.combine_documents import create_stuff_documents_chain

In [ ]:
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
# contextualize_q_prompt

In [ ]:
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [ ]:
qa_prompt.input_variables

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)